# L01: Load and join

**Day 1.** CRISP-DM data understanding: load the departures table and the nested maintenance history, flatten the history, and join it on. The joined table is the working dataset carried to Day 4.

- Reads: data/departures.csv, data/aircraft_history.json
- Writes: work/L01_joined.csv

**Before you start.** Open this notebook from the **Notebooks** folder of the class workspace, next to the **data** and **checkpoints** folders, and run the first two cells.

**How this lab runs.** Work through the cells in order on your own. Each one says what to do with it: **RUN** supplied code, **PREDICT** before running, **DECIDE** by filling a blank, or **WRITE** an answer in words. When most of the room is done, we discuss the answers as a class, then walk through the worked solution together.

### Step 1 · RUN · Run the setup cell

In [ ]:
# RUN · setup: finds the class files and checks that today's are there. Run it first, every time.
from pathlib import Path
# The class files sit in the folder holding "data" and "checkpoints": this notebook's folder or the one above it.
# (On a local install, a folder named AIDSA in your user folder also works.)
_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents, Path.home() / "AIDSA"] if (p / "data").is_dir() and (p / "checkpoints").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Class files not found: open this notebook from the Notebooks folder of the class workspace.")
DATA, CHECKPOINTS, WORK = ROOT / "data", ROOT / "checkpoints", ROOT / "work"
NEEDED = ["data/aircraft_history.json",
          "data/departures.csv"]
missing = [name for name in NEEDED if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError("Not in the workspace yet: " + ", ".join(missing) + ". Ask your instructor: today's files are not in the workspace yet.")
WORK.mkdir(exist_ok=True)
SOURCE = "checkpoint"  # change to "mine" to use your own saved work where it exists
SEED = 2026
import numpy as np
rng = np.random.default_rng(SEED)

def input_path(name):
    """Where this lab reads an earlier lab's output: your own work folder if you chose "mine" and the file is there,
    otherwise the supplied checkpoint."""
    mine = WORK / name
    return mine if SOURCE == "mine" and mine.exists() else CHECKPOINTS / name

print("Found the class files in:", ROOT, "| start from:", SOURCE)

### Step 2 · RUN · Run the helper functions

In [ ]:
# RUN · helper functions: the tested course helpers this lab uses. Run it once, after the setup cell.
# You do not need to read them to do the lab; each is explained where the lab uses it.
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats


def load_departures(path):
    """Read departures.csv exactly as recorded: no cleaning, dates left as text.

    Lab L01.
    """
    return pd.read_csv(path, encoding="utf-8")


def load_history(path):
    """Read aircraft_history.json into a list of per-tail records.

    Lab L01.
    """
    return json.loads(Path(path).read_text(encoding="utf-8"))


def profile_table(df):
    """One row per column: type, missing count and share, distinct values, an example.

    Lab L01.
    """
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_share": df.isna().mean().round(3),
        "distinct": df.nunique(),
        "example": df.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else None),
    })

print("Helper functions ready")

### Step 3 · DO · Read the data dictionary

**Before you begin:** open `data/data_dictionary.md` and read it before any data. It says what one row of each file means, what every field holds, and which values are valid. Every check in this lab is a comparison between what the dictionary promises and what the file contains.

**The question running through the week:** which departures of this airlift and tanker group are
at risk of a maintenance delay, what drives that risk, and what should the group do about it?
This lab is CRISP-DM's *data understanding* phase: before any modelling, find out what the data
actually contains.

## Part 1: Flat findings table

**Specification.** Load both sources and profile the departures table. Then flatten the nested maintenance history
into a table with one row per finding, carrying its tail and inspection.

### Step 4 · RUN · Load both files and profile the departures

In [ ]:
# RUN · supplied code: read it, then run it.
import pandas as pd

# Load exactly as recorded: cleaning comes later, once we know what needs cleaning.
departures = load_departures(DATA / "departures.csv")
history = load_history(DATA / "aircraft_history.json")
print(f"departures: {len(departures):,} rows | history: {len(history)} tail records")

# One row per column: what type pandas inferred, how much is missing, how many distinct values.
profile_table(departures)

### Step 5 · RUN · Look inside one history record

In [ ]:
# RUN · supplied code: read it, then run it.
# The history is nested three levels deep: tail -> inspections -> findings. Look at one record.
first = history[0]
print("tail:", first["tail_id"], "| inspections:", len(first["inspections"]))
print("first inspection:", {k: v for k, v in first["inspections"][0].items() if k != "findings"})
print("its findings:", first["inspections"][0]["findings"][:2])

### Step 6 · PREDICT · Predict the flat table

**PREDICT · before you run the next cell**. The history holds 122 tail records and 293 inspections. How many rows will the flat table have:
one per tail, one per inspection, or one per finding? Will every tail appear in it?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 7 · RUN · Flatten the history

In [ ]:
# RUN · supplied code: read it, then run it.
# record_path walks down to the list we want one row per; meta carries fields from the levels above.
findings = pd.json_normalize(
    history,
    record_path=["inspections", "findings"],
    meta=["tail_id", ["inspections", "inspection_id"], ["inspections", "date"]],
)
n_inspections = sum(len(r["inspections"]) for r in history)
print(f"flat findings: {len(findings)} rows, {findings['tail_id'].nunique()} tails "
      f"(of {len(history)}), from {n_inspections} inspections")
findings.head()

### Step 8 · WRITE · What disappeared?

**WRITE · your interpretation**. What disappeared when the history was flattened, and why does it matter for the join in Part 2?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 2: Joined table

**Specification.** Count each tail's open findings by severity, join the counts onto every departure, and prove
the join did not change the number of departures. Every departure must stay in the table: this is
the working dataset for the week, and a departure that falls out of it is never analysed again.

### Step 9 · RUN · Count each aircraft's open findings

In [ ]:
# RUN · supplied code: read it, then run it.
# Only findings still open on 31 December can affect this year's departures.
open_counts = (findings[findings["status"] == "open"]
               .groupby(["tail_id", "severity"]).size()      # count per tail and severity
               .unstack(fill_value=0)                        # one row per tail, a column per severity
               .rename(columns={"major": "open_major", "minor": "open_minor"})
               .reset_index())
print(f"tails with at least one open finding: {len(open_counts)}")
open_counts.head()

### Step 10 · PREDICT · Predict the join

**PREDICT · before you run the next cell**. departures has 3,030 rows, and every one of them must stay in the working dataset. After the join,
how many rows should the joined table have, and which departures will have no counts to attach?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 11 · DECIDE · Choose the join

In [ ]:
# DECIDE · DP-L01-1 · which rows the join keeps.
# Replace ____ with your choice. The choices are taught on the slide "Joins: which rows survive, and how to prove it".
# how= decides which rows survive the join. validate="many_to_one" makes pandas raise an error if a
# tail appeared twice on the right-hand side, which would silently duplicate departures.
joined = departures.merge(open_counts, on="tail_id", how="____", validate="many_to_one")
unmatched = joined["open_major"].isna()
print(f"rows before {len(departures):,}, after {len(joined):,}")
print(f"departures with no open findings: {unmatched.sum():,} rows, {joined.loc[unmatched, 'tail_id'].nunique()} tails")

# No match means no open findings: zero, not unknown.
joined[["open_major", "open_minor"]] = joined[["open_major", "open_minor"]].fillna(0).astype(int)
print(joined[["open_major", "open_minor"]].describe().loc[["mean", "max"]].round(2))

### Step 12 · RUN · Prove nothing was lost

In [ ]:
# RUN · supplied code: read it, then run it.
# The proof, written as a check that stops the notebook if it is false: every departure is still here.
# If this cell stops with an AssertionError, go back to the join above: the choice was wrong.
assert len(joined) == len(departures), "the join changed the number of departures"
print("check passed: every departure is still in the table")

### Step 13 · WRITE · Read the profile again

**WRITE · your interpretation**. The profile in Part 1 showed 13 distinct aircraft types and 3,000 distinct mission identifiers in
3,030 rows. What do those two numbers tell you about the data, before any cleaning?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 14 · RUN · Save the joined table

In [ ]:
# RUN · saves this part's output to work/.
joined.to_csv(WORK / "L01_joined.csv", index=False, encoding="utf-8")
print("saved work/L01_joined.csv:", joined.shape)